# N122 · 后缀数组与LCP

**目标：** 用后缀排序把重复子串问题变成相邻比较。

**先修：** N118, N037, N040。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 倍增秩；后缀数组；Kasai；LCP；不同子串计数；排序成本。

**配套讲解来源：** [同名逐章意见](../../comment/122_suffix_array_lcp.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这章解决的是"重复子串家族"问题：一个字符串里有多少个不同的子串？最长的重复出现子串是哪个？这类问题的暴力做法要把所有 O(n²) 个子串生成出来再比较，字符串一长就爆炸。

后缀数组的思路是把问题"换个摆放方式"：把字符串的所有 n 个后缀按字典序排序，那么每个子串都是某个后缀的前缀，重复出现的子串就会在排序后"挤在一起"。

具体例子：s = 'banana'（n=6）。它的 6 个后缀排好序是 'a'、'ana'、'anana'、'banana'、'na'、'nana'。用这六行算两件事：
1. 不同非空子串有多少个？全部后缀的前缀总数是 6+5+4+3+2+1 = 21，其中重复的部分恰好被"相邻后缀的公共前缀"抵消，答案是 21 − (1+3+0+0+2) = 15。你可以拿 'a' 验证：'a' 同时是 'a'、'ana'、'anana' 的前缀，在计数里被算了 3 次，要扣掉 2 次。
2. 最长重复子串是谁？看相邻后缀的公共前缀最长是多少：'ana' 和 'anana' 公共前缀 'ana' 长 3，所以 'ana' 出现了至少两次（下标 1–3 和 3–5），它就是最长重复子串。

本章三件事：用倍增法把后缀排序（suffix_array）、用 Kasai 算法线性求相邻 LCP（kasai_lcp）、再用两者相减数出不同子串（count_distinct_substrings_sa）。

## 2. 基础知识：先读懂这些词

- **后缀（suffix）**：从某个下标开始一直取到串尾的一段。s[i:] 就是起点为 i 的后缀。长度为 n 的串恰好有 n 个后缀（不算空串）。
- **后缀数组（SA, suffix array）**：一个长度为 n 的数组 sa，它把 n 个后缀的起点下标按"后缀的字典序从小到大"排好。比如 'banana' 的 sa = [5,3,1,0,4,2]，意思是排第 0 名的是 s[5:]='a'，第 1 名是 s[3:]='ana'，以此类推。
- **名次（rank）**：sa 的反查表——rank[i] 回答"起点为 i 的后缀排第几"。sa 和 rank 互为逆映射，一个从名次查起点，一个从起点查名次。
- **倍增法**：直接给后缀排序，每次比较最长要 O(n) 个字符。倍增法的办法是：第一轮只按"单个字符"排名；之后每一轮拿着上一轮的排名，把每个后缀按"前 2w 个字符"重新排名（w 是上一轮的宽度）。它成立的原因是：比较两段长度 2w 的前缀，等价于先比前半段（长度 w，上一轮已出名次）再比后半段（同样是某个位置开始的长度 w 段，上一轮也已有名次）。所以每轮只需对整数对排序，log n 轮后宽度覆盖全串。
- **LCP（最长公共前缀）**：两个串从第一位开始连续相同的部分。本章特指 lcp[r] = 排名 r 的后缀与排名 r+1 的后缀的 LCP 长度，共 n−1 个值。
- **Kasai 算法**：按"起点下标从小到大"的顺序（而不是排名顺序）逐个求 LCP 的技巧。它利用一个性质：若后缀 i 与它的排名邻居 LCP 为 h，那么后缀 i+1（掐掉第一个字符）与它的排名邻居的 LCP 至少是 h−1。于是每次可以把"当前匹配长度"减 1 当作起点继续扩展，总扩展量是线性的。
- **不同子串计数公式**：不同非空子串数 = n(n+1)/2 − Σlcp。直觉是：按排名逐个"入库"后缀，每个后缀贡献它的全部前缀，但与前一名重合的恰好是 LCP 那么多个，扣掉即可。
- **排序成本**：本章用 Python 内置 sort（比较排序）做每轮排序，一轮 O(n log n)，共 O(log n) 轮，总计 O(n log² n)；如果每轮换成计数排序（O(n)），整体可以压到 O(n log n)。notebook 特意强调不要把两者混称。

## 3. 思路推导：从小例子开始

- Step 1：把问题挂到后缀上。"子串 s[i:j] 出现过几次"等价于"有多少个后缀以它为前缀"；排序后，以同一子串为前缀的后缀必然连成一段。于是重复检测被转化成"排序 + 相邻比较"。
- Step 2：倍增排序先手算 'banana' 的第一轮（width=1）。初始名次就是字符编码：b=98, a=97, n=110，即 ranks=[98,97,110,97,110,97]。排序的键是 (本字符名次, 下一个字符名次，越界用 −1)：i=5 是 (97,−1) 最小；i=1 和 i=3 都是 (97,110) 并列；i=0 是 (98,97)；i=2 和 i=4 都是 (110,97) 并列。本轮得到 sa=[5,1,3,0,2,4]，名次 ranks=[2,1,3,1,3,0]，还有并列（最大名次 3 ≠ 5），继续。
- Step 3：第二轮（width=2）。键变成 (ranks[i], ranks[i+2] 或 −1)：i=5 是 (0,−1)、i=3 是 (1,0)、i=1 是 (1,1)、i=0 是 (2,3)、i=4 是 (3,−1)、i=2 是 (3,3)。排序得 sa=[5,3,1,0,4,2]，重新编号后名次为 [3,2,5,1,4,0]，最大名次 5 = n−1，说明所有后缀名次互不相同、顺序已完全确定，算法提前结束。最终 sa=[5,3,1,0,4,2]，与断言一致。
- Step 4：Kasai 求 LCP。关键性质用 'banana' 看最清楚：按起点顺序处理到 i=3（后缀 'ana'，排名 1）时，与排名邻居 s[1:]='anana' 逐字符比出 LCP=3；接下来处理 i=4（后缀 'na'，正好是 'ana' 掐掉首字符），它与自己排名邻居 s[2:]='nana' 的 LCP 不必从 0 比起——把上次的 3 减 1 得 2 当下界，果然 'na' 与 'nana' 的 LCP 正好是 2，一次新比较都不用多做。
- Step 5：全部跑完得到下面这张表（就是 notebook “运行示例”部分 生成的表）：

| 排名 r | 后缀起点 i | 后缀 | 与下一后缀LCP lcp[r] |
|---|---|---|---|
| 0 | 5 | a | 1 |
| 1 | 3 | ana | 3 |
| 2 | 1 | anana | 0 |
| 3 | 0 | banana | 0 |
| 4 | 4 | na | 2 |
| 5 | 2 | nana | −（最后一名没有下一个） |

- Step 6：数不同子串。按排名把后缀一个个入库：'a' 贡献 1 个新前缀（扣掉与下一名的重复——注意公式统一在"全部前缀总数"里扣 Σlcp）；全部前缀共 21 个，相邻重复共 1+3+0+0+2 = 6 个，答案 21 − 6 = 15，与 notebook 打印一致。

## 4. 核心代码：suffix_array

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def suffix_array(s):
    n = len(s)
    sa = list(range(n))
    ranks = [ord(c) for c in s]
    width = 1
    while width < n:
        sa.sort(key=lambda i: (ranks[i], ranks[i + width] if i + width < n else -1))
        new = [0] * n
        for j in range(1, n):
            a, b = (sa[j - 1], sa[j])
            left = (ranks[a], ranks[a + width] if a + width < n else -1)
            right = (ranks[b], ranks[b + width] if b + width < n else -1)
            new[b] = new[a] + (left != right)
        ranks = new
        if not n or ranks[sa[-1]] == n - 1:
            break
        width *= 2
    return sa
```

### 逐段读懂代码

### 4.1 suffix_array：倍增法后缀排序

（对应程序见下方分段实现与完整代码。）

- `sa=list(range(n))` 是待排序的起点下标；`ranks=[ord(c) for c in s]` 拿字符编码当第 0 轮名次，省一次单独排序。
- `sa.sort(key=lambda i:(ranks[i],ranks[i+width] if i+width<n else -1))` 是每轮的核心：把每个后缀按"前半块名次、后半块名次"这个二元组排序；`i+width<n` 的三目表达式处理后半块越界的情形——越界说明这个后缀本身不足 2w 长，用 −1 补位（比任何名次都小，因为短的先结束就应该排前面，这与字典序一致）。
- 下面那个 for 循环把"排序后的名次"重新编号成 0..n−1 的紧凑整数：遍历排序结果中相邻的 a、b，键对相同就同分（`new[b]=new[a]+(left!=right)` 里 `left!=right` 是布尔值，False 当 0、True 当 1，这句紧凑写法等价于"相同则同号、不同则加一"）。
- `if not n or ranks[sa[-1]]==n-1: break` 是提前终止：`ranks[sa[-1]]` 是最后一名的名次，它等于 n−1 说明名次没有任何并列、所有后缀已分出高下；`not n` 挡住空串（否则 `sa[-1]` 会对空列表取下标直接报错）。
- `width*=2` 让下一轮的比较宽度翻倍。while 条件 `width<n` 保证宽度覆盖全串后不再空转。

### 4.2 kasai_lcp：线性求相邻 LCP

（对应程序见下方分段实现与完整代码。）

- `if len(sa)!=n or set(sa)!=set(range(n)): raise ValueError(...)` 是输入体检：要求 sa 恰好是 0..n−1 的一个排列，防止调用者传垃圾进来让后面悄悄算错。注意它只验证"是排列"，不验证"排好了序"（那是 O(n log n) 的事，本章明确不做）。
- `for r,i in enumerate(sa): rank[i]=r` 一行建好反查表 rank。
- `lcp=[0]*max(0,n-1)` 里那个 `max(0,n-1)` 是在护住 n=0：空串时 max(0,−1)=0，得到空表而不是负长度列表。
- 外层 `for i in range(n)` 是按起点下标顺序走（Kasai 的精髓），`r=rank[i]` 查当前后缀的排名；`if r==n-1: height=0; continue` 处理最后一名——它没有"下一个后缀"可比，把 height 清零后跳过（这一行还能防止下面 sa[r+1] 越界）。
- `j=sa[r+1]` 取排名紧随其后的后缀起点；while 循环从当前 height（带着上一轮剩下的"余温"）开始逐字符比较，能比就 height 加一。
- `height=max(0,height-1)` 是算法的灵魂：为下一个起点 i+1 预置下界 height−1。为什么可以？后缀 i+1 是后缀 i 掐掉一个字符；若后缀 i 与某后缀有长度 h 的公共前缀，掐头后至少还有 h−1。用 max(0,·) 是防 h=0 时减成 −1。
- 摊还视角：height 每个字符最多涨 1、每次循环结束跌 1，涨幅总量不超过 n，所以 while 的总比较次数是 O(n)。

### 4.3 count_distinct_substrings_sa：公式一行算不同子串

```python
def count_distinct_substrings_sa(s):
    sa = suffix_array(s)
    return len(s) * (len(s) + 1) // 2 - sum(kasai_lcp(s, sa))
```

- 第一段 `len(s)*(len(s)+1)//2` 是全部后缀的前缀总数（等差数列求和），第二段 `sum(kasai_lcp(s,sa))` 是排序后相邻后缀的重复前缀总量，两者相减就是不同非空子串数。空串情形：0*(1)//2 − sum([]) = 0，自然正确。

## 5. 分段实现：按顺序运行

**本章接口：** `suffix_array(s)`、`kasai_lcp(s, sa)`、`count_distinct_substrings_sa(s)`

### suffix_array

In [1]:
def suffix_array(s):
    n = len(s)
    sa = list(range(n))
    ranks = [ord(c) for c in s]
    width = 1
    while width < n:
        sa.sort(key=lambda i: (ranks[i], ranks[i + width] if i + width < n else -1))
        new = [0] * n
        for j in range(1, n):
            a, b = (sa[j - 1], sa[j])
            left = (ranks[a], ranks[a + width] if a + width < n else -1)
            right = (ranks[b], ranks[b + width] if b + width < n else -1)
            new[b] = new[a] + (left != right)
        ranks = new
        if not n or ranks[sa[-1]] == n - 1:
            break
        width *= 2
    return sa

### kasai_lcp

In [2]:
def kasai_lcp(s, sa):
    n = len(s)
    if len(sa) != n or set(sa) != set(range(n)):
        raise ValueError('sa must be a permutation')
    rank = [0] * n
    for r, i in enumerate(sa):
        rank[i] = r
    lcp = [0] * max(0, n - 1)
    height = 0
    for i in range(n):
        r = rank[i]
        if r == n - 1:
            height = 0
            continue
        j = sa[r + 1]
        while i + height < n and j + height < n and (s[i + height] == s[j + height]):
            height += 1
        lcp[r] = height
        height = max(0, height - 1)
    return lcp

### count_distinct_substrings_sa

In [3]:
def count_distinct_substrings_sa(s):
    sa = suffix_array(s)
    return len(s) * (len(s) + 1) // 2 - sum(kasai_lcp(s, sa))

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

s='banana'; sa=suffix_array(s); lcp=kasai_lcp(s,sa)
show_table(['排名','后缀起点','后缀','与下一后缀LCP'],[(r,i,s[i:],lcp[r] if r<len(lcp) else '-') for r,i in enumerate(sa)])
print('不同非空子串',count_distinct_substrings_sa(s))

排名,后缀起点,后缀,与下一后缀LCP
0,5,a,1
1,3,ana,3
2,1,anana,0
3,0,banana,0
4,4,na,2
5,2,nana,-


不同非空子串 15


## 7. 正确性、适用条件与复杂度

两个长度2w片段的顺序由其前后两块长度w的排名决定，归纳得到完整后缀顺序。加入字典序下一后缀时，和所有旧后缀共享的最长前缀由相邻前驱给出；扣除相邻LCP总和即可去重。Kasai把前一匹配长度减1作为下一次下界，累计字符扩展线性。

**代价：** Python比较排序倍增O(n log² n)时间O(n)辅助，不误称O(n log n)。给定合法SA的Kasai O(n)时间空间；本实现额外线性验证排列，但不重新验证SA排序。非空不同子串数量为n(n+1)/2−∑LCP。

### 展开理解

正确性分三块说。

倍增排序为什么对：我们逐轮证明"第 k 轮结束后，名次顺序等于按前 2^k 个字符比较的顺序"。第一轮按单字符排，显然成立。往后每一轮，比较两个后缀的前 2w 个字符等价于"先比前 w 个字符、相同再比第 w 到 2w 个字符"，而这两段各自的相对顺序上一轮都已经编进名次里了，所以对整数二元组排序等价于对长前缀排序。宽度翻倍最终超过 n，名次就变成完整后缀的顺序。

Kasai 为什么对：一句话——"掐头不减太多"。设排名里紧跟后缀 i 之后的是后缀 j，两者公共前缀长 h。现在看后缀 i+1（i 掐头）和"某个"公共前缀至少为 h−1 的后缀：具体地说，j 掐头得到的 j+1 的名次邻居与 i+1 的公共前缀不短于 h−1（因为 i+h 与 j+h 之前的部分全相同）。所以把 height 从 h−1 起跳再向上比，算出的就是真实的 LCP，不会漏也不会错。累计起来字符比较总量是线性的，因为 height 每轮最多涨 1，跌 1 是免费的。

计数公式为什么对：把 n 个后缀按排名从 0 到 n−1 依次"入库"，每个后缀带来"它的长度"个前缀（这些前缀互不相同），但它与前一名后缀共享的前缀恰好是 LCP 那么多个、是早已入库的旧货，要扣除。相邻的扣完，非相邻的重复也自动被扣干净了（共享前缀长的两个后缀在排序中不可能被隔开比 LCP 更远），所以总数就是 n(n+1)/2 − Σlcp。

复杂度（拿具体 n 感受）：
- suffix_array：一共约 log₂n 轮，每轮一次比较排序 O(n log n)，总计 O(n log² n)，额外空间 O(n)。n=10⁵ 时约 10⁵ × 17 × 17 ≈ 3×10⁷ 量级的比较，Python 能跑（这也是 notebook 强调"不误称 O(n log n)"的原因——用计数排序替换每轮的 sort 才是 O(n log n)）。
- kasai_lcp：一次体检 O(n)，主循环的字符比较摊还 O(n)，总计 O(n) 时间空间。n=10⁵ 时就是十万次上下的比较。
- count：就是两个 O(n log² n) 以下的部件之和，瓶颈仍在排序。

## 8. 单元测试：每个用例在检查什么

```python
assert suffix_array('banana') == [5, 3, 1, 0, 4, 2]
```
这条是教科书标准用例：'banana' 自带大量重复子串（a、na、ana），能同时考验排序正确性和并列名次的重新编号。预期 [5,3,1,0,4,2] 对应 'a','ana','anana','banana','na','nana'，你可以对着第三节那张表逐行核对。

```python
assert suffix_array('') == [] and kasai_lcp('', []) == []
```
这条专测空串边界：空串没有后缀，SA 应为空表、LCP 也应为空表。它顺带检验了代码里 `max(0,n-1)` 和 `not n` 两个护身符。

```python
for n in range(8):
    for chars in product('ab', repeat=n):
        ...
```
这段又是"对拍"：枚举所有长度 0–7 的 a/b 串，三件事逐一对照暴力解——SA 必须严格等于"按下标以 s[i:] 排序"的直接写法；不同子串数必须等于把全部子串塞进集合后的大小；每个 lcp[r] 必须等于"取两个相邻后缀、逐位比较出的最长公共前缀长度"。a/b 字母表最小、重复最密，是把并列、重复子串这些坑全踩一遍的极端采样。

In [5]:
assert suffix_array('banana') == [5, 3, 1, 0, 4, 2]

assert suffix_array('') == [] and kasai_lcp('', []) == []

from itertools import product

for n in range(8):
    for chars in product('ab', repeat=n):
        s = ''.join(chars)
        sa = suffix_array(s)
        lc = kasai_lcp(s, sa)
        assert sa == sorted(range(n), key=lambda i: s[i:])
        assert count_distinct_substrings_sa(s) == len({s[i:j] for i in range(n) for j in range(i + 1, n + 1)})
        for r, h in enumerate(lc):
            a, b = (s[sa[r]:], s[sa[r + 1]:])
            assert h == max((k for k in range(min(len(a), len(b)) + 1) if a[:k] == b[:k]))

print('N122: 所有本章断言通过')

N122: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 从LCP求最长重复子串。

**练习 2：** 区分用比较排序的O(nlog²n)实现与计数排序版复杂度。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（从 LCP 求最长重复子串）**：方向提示——一个子串重复出现，等价于它是两个不同后缀的公共前缀；而在排好序的后缀里，某段公共前缀的最长者在相邻位置就能取到（共享前缀越长的两个后缀在排序里贴得越近）。所以答案就是 max(lcp)：'banana' 里 max=3 出现在 r=1，对应的公共前缀 s[sa[1]:sa[1]+3] = 'ana'。你要写清楚"为什么看相邻就够了"，再拿一个手算例子（比如 'abcbcba'）验证。边界别忘了"没有重复子串"时应返回空串（max(lcp) 为 0 或 n≤1）。

**练习 2（比较排序版与计数排序版的复杂度区分）**：提示——数一数每轮排序的代价：Python 的 sort 是比较排序，n 个键要 O(n log n)，乘上 log n 轮就是 O(n log² n)；计数排序每轮按"二元组名次"分桶，是 O(n)，乘上 log n 轮就是 O(n log n)。你可以做个小实验：固定几个 n（如 10³、10⁴），分别计时两个版本，看比值是否接近 log n。写清楚结论时注意口径：本章接口属于前者，别把功劳记错。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
def suffix_array(s):
    n = len(s)
    sa = list(range(n))
    ranks = [ord(c) for c in s]
    width = 1
    while width < n:
        sa.sort(key=lambda i: (ranks[i], ranks[i + width] if i + width < n else -1))
        new = [0] * n
        for j in range(1, n):
            a, b = (sa[j - 1], sa[j])
            left = (ranks[a], ranks[a + width] if a + width < n else -1)
            right = (ranks[b], ranks[b + width] if b + width < n else -1)
            new[b] = new[a] + (left != right)
        ranks = new
        if not n or ranks[sa[-1]] == n - 1:
            break
        width *= 2
    return sa

def kasai_lcp(s, sa):
    n = len(s)
    if len(sa) != n or set(sa) != set(range(n)):
        raise ValueError('sa must be a permutation')
    rank = [0] * n
    for r, i in enumerate(sa):
        rank[i] = r
    lcp = [0] * max(0, n - 1)
    height = 0
    for i in range(n):
        r = rank[i]
        if r == n - 1:
            height = 0
            continue
        j = sa[r + 1]
        while i + height < n and j + height < n and (s[i + height] == s[j + height]):
            height += 1
        lcp[r] = height
        height = max(0, height - 1)
    return lcp

def count_distinct_substrings_sa(s):
    sa = suffix_array(s)
    return len(s) * (len(s) + 1) // 2 - sum(kasai_lcp(s, sa))

# 示例与回归测试
assert suffix_array('banana') == [5, 3, 1, 0, 4, 2]

assert suffix_array('') == [] and kasai_lcp('', []) == []

from itertools import product

for n in range(8):
    for chars in product('ab', repeat=n):
        s = ''.join(chars)
        sa = suffix_array(s)
        lc = kasai_lcp(s, sa)
        assert sa == sorted(range(n), key=lambda i: s[i:])
        assert count_distinct_substrings_sa(s) == len({s[i:j] for i in range(n) for j in range(i + 1, n + 1)})
        for r, h in enumerate(lc):
            a, b = (s[sa[r]:], s[sa[r + 1]:])
            assert h == max((k for k in range(min(len(a), len(b)) + 1) if a[:k] == b[:k]))

print('N122: 所有本章断言通过')

N122: 所有本章断言通过


## 11. 代表题与迁移

- **1044. Longest Duplicate Substring（延伸题）**：给一个串，找最长且出现至少两次的子串。这题练的正是本章"SA + LCP"组合拳：排序后缀、扫一遍 lcp 取最大值即可（官方期望解还有二分长度 + Rabin-Karp 的路线，可以拿来和 SA 路线对比）。它就是练习 1 的完整版。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。